# Lab 3.3 — Tool Loops and Caching: Converse → Mantle

**Duration:** ~8 min · **Level:** L300 · **Lab 3 of 4 — part 3/4**

We'll take a full working Converse tool loop and migrate it, step by step,
to Mantle Chat Completions. The goal is a diff you can copy into your own
codebase.


In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

import boto3
from src.common.mantle import openai_client, GPT_OSS_120B

runtime = boto3.client("bedrock-runtime")
mantle  = openai_client()

# Mini "tool" shared across both runs.
_PRICES = {"p1": 899, "p2": 1199, "p3": 2399}
def get_price(product_id: str) -> dict:
    if product_id not in _PRICES:
        return {"error": f"unknown product {product_id!r}"}
    return {"product_id": product_id, "usd": _PRICES[product_id]}


## 1. Before — Converse tool loop

Converse shape:

- Tools declared under `toolConfig.tools[].toolSpec`.
- `toolChoice={"auto": {}}` / `{"any": {}}` / `{"tool": {"name": "..."}}`.
- Tool calls arrive as `toolUse` blocks with **`input`** already a dict.
- Tool results go back in a `user` message as `toolResult` blocks.
- Loop terminates when `stopReason != "tool_use"`.


In [ ]:
CONVERSE_MODEL_CANDIDATES = [
    "anthropic.claude-haiku-4-5",
    "mistral.mistral-large-2407-v1:0",
    "anthropic.claude-3-5-haiku-20241022-v1:0",
]

converse_messages = [{
    "role": "user",
    "content": [{"text": "What do products p1 and p3 cost?"}]
}]

converse_ok = False
for cand in CONVERSE_MODEL_CANDIDATES:
    try:
        for _ in range(5):
            r = runtime.converse(
                modelId=cand,
                messages=converse_messages,
                toolConfig={
                    "tools": [{
                        "toolSpec": {
                            "name": "get_price",
                            "description": "Look up a product's USD price.",
                            "inputSchema": {"json": {
                                "type": "object",
                                "properties": {"product_id": {"type": "string"}},
                                "required": ["product_id"],
                            }},
                        }
                    }],
                    "toolChoice": {"auto": {}},
                },
                inferenceConfig={"maxTokens": 400, "temperature": 0.2},
            )
            assistant = r["output"]["message"]
            converse_messages.append(assistant)
            if r["stopReason"] != "tool_use":
                text = next((b["text"] for b in assistant["content"] if "text" in b), "")
                print(f"[{cand}] final:", text)
                converse_ok = True
                break
            results = []
            for b in assistant["content"]:
                if "toolUse" not in b:
                    continue
                tu = b["toolUse"]
                out = get_price(**tu["input"])   # input is already a dict
                results.append({"toolResult": {
                    "toolUseId": tu["toolUseId"],
                    "content": [{"json": out}],
                    "status": "success",
                }})
            converse_messages.append({"role": "user", "content": results})
        if converse_ok:
            break
    except Exception as e:
        print(f"  {cand}: {e.__class__.__name__}")

if not converse_ok:
    print("⚠️  No Converse model available — skipping baseline.")


## 2. After — Chat Completions tool loop

Spot the diff:

- Tool wrapper changes (`toolSpec` → `function`).
- `tool_choice` vocabulary changes (`{"any": {}}` → `"required"`).
- **`tool_calls[].function.arguments` is a JSON string** — you must
  `json.loads()` it.
- Tool results go back as a `role: "tool"` message with `tool_call_id`
  and **string** content.
- Loop terminates when `finish_reason != "tool_calls"`.


In [ ]:
cc_tools = [{
    "type": "function",
    "function": {
        "name": "get_price",
        "description": "Look up a product's USD price.",
        "parameters": {
            "type": "object",
            "properties": {"product_id": {"type": "string"}},
            "required": ["product_id"],
            "additionalProperties": False,
        },
    },
}]

cc_messages = [{"role": "user", "content": "What do products p1 and p3 cost?"}]

for _ in range(5):
    r = mantle.chat.completions.create(
        model=GPT_OSS_120B,
        messages=cc_messages,
        tools=cc_tools,
        tool_choice="auto",
        max_tokens=400,
        temperature=0.2,
    )
    msg = r.choices[0].message
    assistant_turn = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:
        assistant_turn["tool_calls"] = [tc.model_dump() for tc in msg.tool_calls]
    cc_messages.append(assistant_turn)
    if r.choices[0].finish_reason != "tool_calls":
        print("final:", msg.content)
        break
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)     # STRING → dict
        out  = get_price(**args)
        cc_messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(out),              # STRING
        })
else:
    print(f"loop hit iteration cap without terminating")


## 3. The migration diff — line-by-line

| Converse line | Chat Completions replacement |
|---|---|
| `toolConfig={"tools":[...], "toolChoice":{"auto":{}}}` | `tools=[...], tool_choice="auto"` |
| `"toolSpec": {"inputSchema": {"json": {...}}}` | `"function": {"parameters": {...}}` |
| `{"toolChoice": {"any": {}}}` | `tool_choice="required"` |
| `{"toolChoice": {"tool": {"name": "x"}}}` | `tool_choice={"type":"function","function":{"name":"x"}}` |
| `stopReason == "tool_use"` | `finish_reason == "tool_calls"` |
| `toolUse["input"]` (dict) | `json.loads(tc.function.arguments)` (string → dict) |
| `{"toolResult":{"toolUseId":…,"content":[{"json":…}]}}` in a **user** msg | `{"role":"tool","tool_call_id":…,"content": json.dumps(…)}` |
| Rich content blocks (image/doc/JSON) in tool results | **String only** — lossy; serialize carefully |

Keep this table open when you're migrating a codebase. Every row is a
place where the compiler won't save you — behaviour will silently differ.


## 4. Caching — `additionalModelRequestFields` → `extra_body`

Converse uses `additionalModelRequestFields` to pass provider-specific
extensions. On Mantle, the equivalent is `extra_body`. Example:


In [ ]:
# Before (Converse, illustrative — don't run unless your model supports it):
#
# runtime.converse(
#     modelId="anthropic.claude-sonnet-4-20250514-v1:0",
#     ...,
#     additionalModelRequestFields={"thinking": {"type": "enabled", "budget_tokens": 1024}},
# )

# After (Mantle Chat Completions):
r = mantle.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[{"role": "user", "content": "Pick a number and explain."}],
    max_tokens=60,
    extra_body={
        "cache_salt": "migration-demo-v1",      # prefix-cache affinity hint
        # Other provider-extension fields go here. Availability is per-model;
        # verify against the model card before committing to a flag. Examples:
        #   "thinking": {"type": "adaptive"}      # Claude extended thinking
        #   "reasoning": {"effort": "medium"}    # OpenAI reasoning models
    },
)
print(r.choices[0].message.content.strip())


## 5. Checklist

When you touch a Converse tool loop in production code, do this *in order*:

- [ ] Build a local unit test against the **current** Converse shape — lock
      it in as the baseline.
- [ ] Migrate the schema wrapper.
- [ ] Migrate the `toolChoice` → `tool_choice` mapping.
- [ ] Add `json.loads()` around every `tool_calls[].function.arguments`.
- [ ] Switch the tool-result message to `role:"tool"` + `tool_call_id`.
- [ ] Flatten rich tool-result content to strings.
- [ ] Update the loop termination predicate.
- [ ] Migrate `additionalModelRequestFields` → `extra_body`.
- [ ] Re-run the unit test — should still pass (same output given same input).
- [ ] Shadow-run the new loop against prod traffic for 48 h before cutover.
